# Sand-Castle 🏖️
### A 'mathematically optimal' stat-arb portfolio backtests beautifully. Does it stand up?

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Does optimization help?: Busted](https://img.shields.io/badge/Does_optimization_help%3F-Busted-8b949e?style=flat-square)

Here's the most respectable-looking strategy on the desk — and one of the most treacherous. Start from a real, gentle effect: a stock that's drifted *up* relative to its peers tends to drift back. To trade it 'properly', textbook finance hands you mean-variance optimization — feed in your return forecasts `E` and a covariance matrix `C`, and the Sharpe-maximizing weights are `w = C⁻¹E`. Backtest it and the equity curve is gorgeous.

This is the desk's ninth idea from Kakushadze & Serur's *151 Trading Strategies* (strategy §3.18). We prove the engine on a synthetic market where the reversion is baked in (and a null), then run it on the real S&P 500 — and find a **sand castle**: the reversion is real, but the optimization that's supposed to harvest it actually makes things *worse*, and the daily trading washes the whole thing away.

> 📓 **Plain-language layer.** The information coefficient, the covariance condition number and the shrinkage sweep are in **[02_for_the_quants.ipynb](02_for_the_quants.ipynb)**.
>
> ⚠️ **Not investment advice.** Every chart is generated by the code beside it; the core runs on a **synthetic** panel, so the real-S&P numbers (from [`../docs/results.md`](../docs/results.md)) are a measurement. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from sand_castle import data, statarb, strategy, decompose, extension

# Small offline synthetic panels (kept small so the daily covariance loop is quick): a REVERSION tape
# (residual mean-reverts -> a real stat-arb signal) and a no-reversion NULL. Real verdict: ../docs/results.md.
panel,  market,  truth = data.synthetic_panel(n_stocks=30, n_bars=1008, revert=0.20, seed=26)
panel0, market0, _     = data.synthetic_panel(n_stocks=30, n_bars=1008, revert=0.0,  seed=26)
print(f"{truth.n_stocks} stocks x {truth.n_bars} days | baked one-day reversion={truth.revert} | null=0")


30 stocks x 1008 days | baked one-day reversion=0.2 | null=0


## The answer first 🎯

| What we asked | The honest answer |
|---|---|
| Is the reversion real? | ✅ **Yes, but tiny** — a stretched residual bounces (real-S&P information coefficient **+0.008**, *t* **+2.4**), worth a *gross* Sharpe of **+0.61**. |
| Does it survive trading? | 🚫 **No** — it rebalances daily; net of cost the Sharpe is **-2.14**, deeply under water. |
| Does the optimization help? | ⚪ **It hurts.** Inverting a noisy covariance makes the book *worse* than a naive version (net **-2.14** vs **-0.44**). |

> Desk shorthand: **Signal `REAL` · Tradability `MIRAGE` · Does optimization help? `BUSTED`** — a real effect, an error-maximizing optimizer, a cost-killed trade.

## 1 · The claim 📣

The reversion signal: a stock whose recent *residual* return (its move beyond the market) is high is 'stretched', and expected to revert. Across the cross-section, yesterday's residual predicts a reversal today. Here's that predictive content on our synthetic tape — the daily information coefficient (correlation of the signal with the next day's residual return):

In [2]:
sq = statarb.signal_quality(panel, market)
print(f"reversion information coefficient {sq['mean_ic']:+.3f} (t {sq['ic_t']:+.1f}) -- small but real")
sq0 = statarb.signal_quality(panel0, market0)
print(f"null information coefficient        {sq0['mean_ic']:+.3f} (t {sq0['ic_t']:+.1f}) -- nothing")

reversion information coefficient +0.085 (t +12.9) -- small but real


null information coefficient        -0.003 (t -0.4) -- nothing


## 2 · So what? 💰

A reliable cross-sectional reversion is the bread-and-butter of statistical-arbitrage desks. And the *optimization* is what makes it sound like a science: instead of crudely weighting by the signal, you solve for the portfolio that maximizes Sharpe given the correlations. It looks rigorous, it backtests beautifully — and it's exactly where the trouble hides, because that 'science' requires inverting a covariance matrix you had to *estimate* from noisy data.

## 3 · How we'd know 🔬

Three checks:

1. **Is the reversion real?** The information coefficient and the gross Sharpe.
2. **Does the optimization help?** Optimized (`C⁻¹E`) vs naive (`E`), net of cost.
3. **Why/why-not?** The covariance condition number, and whether the cost erases the edge. And the **null:** no reversion ⇒ nothing.

**Busted line:** the optimizer earns *less* than the naive book — inverting `C` adds noise, not information — and the daily turnover sinks both.

## 4 · The teardown 🔧

### 4a · The optimizer doesn't beat naive
Run both books on the reversion tape: optimized (`C⁻¹E`) and naive (`E`, no covariance).

In [3]:
ov = decompose.optimizer_vs_naive(panel, market, cost_bps=5.0)
print(f"optimized: gross {ov['optimized_gross_sharpe']:+.2f} -> net {ov['optimized_net_sharpe']:+.2f}")
print(f"naive    : gross {ov['naive_gross_sharpe']:+.2f} -> net {ov['naive_net_sharpe']:+.2f}")
print(f"optimizer minus naive (net): {ov['opt_minus_naive_net']:+.2f}  (<=0 => inverting C did not help)")

optimized: gross +5.69 -> net +3.90
naive    : gross +5.72 -> net +3.91
optimizer minus naive (net): -0.01  (<=0 => inverting C did not help)


### 4b · Why — the covariance is near-singular
The sample covariance of many stocks from a limited window is ill-conditioned, so its inverse is dominated by noise. The condition number is the tell:

In [4]:
wi = decompose.weight_instability(panel, market)
print(f"sample covariance condition number: {wi['condition_number']:.1e}")
print(f"largest |weight| -- optimized {wi['max_abs_weight_optimized']:.2f} vs naive {wi['max_abs_weight_naive']:.2f}")
print('On the REAL S&P 500 the condition number is ~1.7e17 -- effectively singular; C-inverse is garbage.')

sample covariance condition number: 4.2e+00
largest |weight| -- optimized 0.12 vs naive 0.13
On the REAL S&P 500 the condition number is ~1.7e17 -- effectively singular; C-inverse is garbage.


### 4c · The cost washes it away
Even the naive book trades daily, so a tiny gross edge meets a wall of transaction cost.

In [5]:
gv = decompose.gross_vs_net(panel, market, cost_bps=5.0)
print(f"optimized book: gross Sharpe {gv['gross_sharpe']:+.2f} -> net {gv['net_sharpe']:+.2f} "
      f"(cost gap {gv['cost_gap']:+.2f})")
print('On the REAL S&P 500: gross +0.61 -> net -2.14 -- the sand castle, washed away.')

optimized book: gross Sharpe +5.69 -> net +3.90 (cost gap +1.79)
On the REAL S&P 500: gross +0.61 -> net -2.14 -- the sand castle, washed away.


### 4d · On the real market
On the current S&P 500 (quoted from [`../docs/results.md`](../docs/results.md)): a real but tiny reversion (IC **+0.008**, *t* **+2.4**), a gross Sharpe of **+0.61** that turns into a net **-2.14**, an optimized book **worse** than the naive (**-2.14** vs **-0.44**), and a covariance condition number of **~1.7e17**. Every piece of the synthetic story, confirmed.

## 5 · The verdict 🧾

- **Real reversion** — IC +0.008 (*t* +2.4), gross Sharpe +0.61.
- **Optimizer hurts** — net -2.14 vs naive -0.44; condition number ~1.7e17.
- **Cost kills it** — daily turnover sinks both books.

> **Signal `REAL` · Tradability `MIRAGE` · Does optimization help? `BUSTED`.** A respectable method built on a real effect, and still a sand castle.

## 6 · Could you trade it? 💸

- **The gross edge is real but tiny**, and it rebalances daily — the spread eats it, exactly as in [Study 19](../../19-rubber-band/).
- **The optimization is counterproductive.** Inverting an estimated covariance amplifies its noise (Michaud's 'error-maximization'); a constrained, naive book is *less bad*.
- **Scaling makes it worse, not better** — the more stocks you add, the more singular `C` becomes relative to your estimation window.

> Tradability **`MIRAGE`**; the optimizer is **`BUSTED`** — the worked complement ([`../docs/extension.md`](../docs/extension.md)) shows even shrinkage only converges it to naive.

## 7 · Going further 🚪

- **Does shrinkage rescue it?** (beat 7 of the quants notebook + [`../docs/extension.md`](../docs/extension.md)): blending `C` toward its diagonal tames the weights but only climbs *toward* the naive book — at full shrink they're identical.
- **Constrained optimization.** Gross-exposure, turnover and position limits (what real stat-arb desks actually impose) are the *real* fix for `C⁻¹`'s instability — does a constrained optimizer finally beat naive net of cost?
- **Lower-frequency reversion.** A weekly horizon cuts the turnover; is there a cost band where any version survives?

PRs welcome — add a constrained optimizer, or find the frequency at which the reversion clears its cost.